# [8.8] Blind prediction: changing output gain
For a compiled induction transformer with fixed attention sharpness and weight noise,
what is **Δ planted-edge ROC AUC for EAP = AUC(gain 1) − AUC(gain 10)**?
Commit a number in [−1, 1], confidence, and a causal explanation before either condition runs.

Given `BOS 7 4 2 7`, predict `4`: copy the token after the earlier `7`.
You will compile this computation, score edge interventions, and compare a controlled change.
The only experimental change is unembedding gain 10 → 1. Attention sharpness stays 20;
attention-weight noise σ=0.03, architecture, data, corruption, seeds and mean-answer-logprob
metric stay fixed. The postcommit control removes weight noise at gain 1.

Objectives: write a select/aggregate compiler; distinguish single-edge differences from
edge-mask derivatives and midpoint integrated gradients; implement tie-aware absolute-score
ROC AUC; compare a forecast with paired measurements and task behavior.

**Exposure:** opening equivalent results or the reference lesson makes this practice.
Use `unknown`, `seen`, or `self_reported_not_seen`; no automatic unseen score or mastery badge.
This is an exploratory compiled-model diagnostic, not an independent TEST evaluation.
Expected outputs before commitment below refer only to independent practice fixtures.
No target result or target figure is prefilled.


In [ ]:
from pathlib import Path
import sys
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'mastery_hub').is_dir())
if str(root) not in sys.path:
    sys.path.insert(0,str(root))
from mastery_hub.units.circuit_prediction import lesson
import torch as t
from IPython.display import display, Code
t.set_num_threads(1)
t.set_default_dtype(t.float32)
t.set_default_device('cpu')
CASE = dict(lesson.CASE)
print(CASE)
# Existing installed environment only. No install, pretrained loading, training, or GPU calls.


## Exercise 1: build the compiler in three small steps (20–30 minutes)

### 1a. Selection and copying: `compile_select_aggregate`
Arguments `model, layer, head` identify one head. `query_space`, `key_space`, `value_space`,
`out_space` are half-open residual slices, with widths nq, nk, nv, nv respectively.
`predicate` has shape `(nq,nk)`; nk and nv must fit inside `d_head`. The head's W_Q/W_K/W_V
have shape `(d_model,d_head)` and W_O has shape `(d_head,d_model)`.
Return nothing: write only this head's designated blocks into initially zero weights.
Q embeds predicate × sharpness × √d_head, K and V identity blocks, O amplitude × identity.
Then q Q (k K)^T/√d_head = sharpness × q predicate k^T; V/O copies selected values.
In the 8D DEV fixture, nq=nk=nv=2 and d_head=3: query [1,0] gives scores [2,0].
Implement this helper first. The cell's `select` test checks its four weight blocks without a forward call.
<details><summary>Hint 1a: which axes?</summary>Write W_Q[query_space,:nk],
W_K[key_space,:nk], W_V[value_space,:nv], and W_O[:nv,out_space], all under no_grad.
An identity predicate is only a special case: query and key widths may differ.</details>

### 1b. Compose two layers: `compile_induction`
Start with `utils.empty_attn_only_model(2,3,81,17,16,17,'cpu')`. This utility only allocates
an empty architecture; the program's weights are yours. Residual coordinates are
`tok=0:16`, `pos=16:33`, `prev=33:49`, `out=49:65`, `junk=65:81`.
Write identity token/position embeddings and unembed only out, multiplied by `gain`.
Heads 0.0 and 0.1 select previous position and copy tok→prev. Head 0.2 selects uniformly
and copies tok→junk. Head 1.0 matches tok→prev and copies tok→out. Head 1.1 matches
junk→junk and copies junk→junk; 1.2 matches positions and copies tok→junk. All other
weights stay zero. Use sharpness20, amplitude1. Return the configured model; do not call it.

Hand trace `BOS 7 4 2 7`: at position2, layer0's prev slice contains the earlier token7;
at position4 the query token7 matches that position2 key and copies position2's current token4.
The two previous-token heads add into the same prev slice. Finite softmax is a soft match,
not an exact symbolic selector. The separate `induction` test checks every composed weight block
on a weights-only mock with gain2.5, without running either target gain condition.
<details><summary>Hint 1b: previous-position matrix</summary>Make a 17×17 zero predicate,
then write identity16 into rows1:, columns:-1. Query position q selects key position q−1.
The induction predicate is identity16, between token and previous-token slices.</details>

### 1c. Add controlled noise: `add_weight_noise`
Only W_Q/W_K/W_V/W_O receive Gaussian noise. Use a local generator seeded by the argument,
iterate named parameters in existing order, preserve embedding/unembedding weights and the global RNG.
Return the model. The final `compiler` check includes zero-noise and repeated-seed controls.
<details><summary>Hint 1c: reproducibility</summary>Sample with a local torch.Generator;
noise scale is sigma, independent of gain. A zero sigma must preserve all weights exactly.</details>

Run the cell after each substep: it prints successful local checks before reaching an unfinished
later function. A later NotImplementedError means continue to the next substep, not erase the earlier work.


In [ ]:
COMPILER_SOURCE = '''
def compile_select_aggregate(model, layer, head, query_space, key_space,
                             predicate, value_space, out_space, sharpness=20., amplitude=1.):
    raise NotImplementedError("Implement compile_select_aggregate")


def compile_induction(gain):
    raise NotImplementedError("Implement compile_induction")


def add_weight_noise(model, sigma, seed):
    raise NotImplementedError("Implement add_weight_noise")
'''
ns = lesson.practice_namespace(COMPILER_SOURCE)
print(lesson.practice_test(ns, 'select'))
print(lesson.practice_test(ns, 'induction'))
print(lesson.practice_test(ns, 'compiler'))


Expected: `compiler: independent practice fixture passed`. This does not evaluate the target case.


## Exercise 2: score each valid edge
`graph.run(tokens, corrupt_out, keep)` interpolates each edge between its clean sender
and the corresponding sender on corrupted input. Invalid edges must stay zero.
Exact: m(all edges) − m(all except e). EAP: ∂m/∂keep_e at all valid keep=1.
IG10: average these derivatives at ten midpoint masks `(k+0.5)/10`, k=0…9.
This is the common straight-line **edge-mask** path, not activation-space IG.
<details><summary>Help</summary>Create a fresh leaf mask each time. Use `autograd.grad`
on the scalar metric, detach scores, and retain signs. A negative score can still rank
highly by absolute importance. No model outcome is needed to implement the definitions.</details>


In [ ]:
SCORES_SOURCE = '''
def patching_scores(graph, tokens, corrupt_out, metric):
    raise NotImplementedError("Implement patching_scores")


def eap_scores(graph, tokens, corrupt_out, metric):
    raise NotImplementedError("Implement eap_scores")


def eap_ig_scores(graph, tokens, corrupt_out, metric):
    raise NotImplementedError("Implement eap_ig_scores")
'''
ns = lesson.practice_namespace(SCORES_SOURCE)
print(lesson.practice_test(ns, 'scores'))


Expected: `scores: independent practice fixture passed`. This does not evaluate the target case.


## Exercise 3: rank against planted structure
Restrict scores and labels to valid edges, then take absolute scores. AUC is the mean over
all positive/negative pairs of `1` for positive>negative and `0.5` for ties.
Reject inputs with an empty class. The random ranking expectation is 0.5.
<details><summary>Help</summary>Use broadcasting to compare every positive with every
negative. Ignore invalid entries even if their score is huge.</details>


In [ ]:
AUC_SOURCE = '''
def roc_auc(scores, planted, valid):
    raise NotImplementedError("Implement roc_auc")
'''
ns = lesson.practice_namespace(AUC_SOURCE)
print(lesson.practice_test(ns, 'auc'))


Expected: `auc: independent practice fixture passed`. This does not evaluate the target case.


## Exercise 4: assemble the scientific result visibly
Return JSON-native `auc` and full signed `scores` per method, plus `task_accuracy` and
`answer_logprob`. For clean prompts use logits at positions `9:16`, targets `clean[:,10:17]`;
mean over both prompts and predictable positions. Accuracy is argmax agreement;
answer logprob is gathered log-softmax. Do not change the denominator by condition.
<details><summary>Help</summary>Detach tensors before converting to lists. The fixture is
an invented three-token classifier, unrelated to either benchmark outcome.</details>


In [ ]:
ASSEMBLY_SOURCE = '''
def assemble_result(scores, graph, planted, logits, clean):
    raise NotImplementedError("Implement assemble_result")
'''
ns = lesson.practice_namespace(ASSEMBLY_SOURCE + "\n" + AUC_SOURCE)
print(lesson.practice_test(ns, 'assembly'))


Expected: `assembly: independent practice fixture passed`. This does not evaluate the target case.


## Exposed DEV ladder — select, then copy (8–12 minutes)
Before freezing the exact target, use only this one-head, two-key arithmetic fixture.
It has residual width 8 and head width 3: query/key/value/output each occupy two coordinates.
With query [1,0], identity predicate, sharpness 2 and amplitude .7, hand-predict both
output coordinates. A hard selector would give [.7,0]; finite softmax need not.

Write your forecast, then choose an explanation: **A:** amplitude controls what gets selected;
**B:** Q/K and sharpness control selection, while V/O amplitude scales the copy.
Which paired observation would rule A out? Compare .7 copy, zero copy and uniform selection.
This is public DEV, not either gain condition, not an EAP/AUC target estimate.
Numeric confidence is an **unscored self-rating** of your reasoning, not P(a number is exactly right).


In [ ]:
dev_forecast = [.7, 0.]  # Worked hard-selection approximation; replace before running.
dev_self_rating = .6  # unscored self-rating, not a calibrated probability
dev_explanation = 'B'
dev_disconfirming_observation = 'Amplitude changes output but leaves attention unchanged.'


In [ ]:
from types import SimpleNamespace
import matplotlib.pyplot as plt
dev_ns = lesson.practice_namespace(COMPILER_SOURCE)
def dev_select_copy(query, amplitude, uniform=False):
    # No transformer constructor or target call: only small public weight arrays.
    attn = SimpleNamespace(**{name: t.zeros(shape) for name,shape in
        [('W_Q',(1,8,3)),('W_K',(1,8,3)),('W_V',(1,8,3)),('W_O',(1,3,8))]})
    model = SimpleNamespace(blocks=[SimpleNamespace(attn=attn)], cfg=SimpleNamespace(d_head=3))
    predicate = t.zeros(2,2) if uniform else t.eye(2)
    dev_ns['compile_select_aggregate'](model,0,0,slice(0,2),slice(2,4),predicate,
                                     slice(4,6),slice(6,8),2.,amplitude)
    residual = t.zeros(2,8)
    residual[:,2:4] = t.eye(2); residual[:,4:6] = t.eye(2)
    q = t.zeros(8); q[:2] = t.tensor(query)
    scores = (q @ attn.W_Q[0]) @ (residual @ attn.W_K[0]).T / 3**.5
    attention = scores.softmax(-1)
    output = (attention @ (residual @ attn.W_V[0]) @ attn.W_O[0])[6:8]
    return attention, output
dev_pairs = {name: dev_select_copy([1.,0.], amplitude, uniform) for name,amplitude,uniform in
             [('copy .7',.7,False),('zero copy',0.,False),('uniform selection',.7,True)]}
fig,axes = plt.subplots(1,2,figsize=(10,3),layout='constrained')
for name,(attention,output) in dev_pairs.items():
    axes[0].plot([0,1],attention.tolist(), 'x--' if name=='zero copy' else 'o-',label=name)
    axes[1].plot([0,1],output.tolist(),'o-',label=name)
axes[0].set(title='DEV selection',xlabel='Key',ylabel='Attention probability',xticks=[0,1])
axes[1].set(title='DEV copied coordinates',xlabel='Output coordinate',ylabel='Activation',xticks=[0,1])
for ax in axes: ax.legend()
display(fig)
print('DEV copy:',dev_pairs['copy .7'][1].tolist())
print('DEV coordinate absolute forecast errors:',(dev_pairs['copy .7'][1]-t.tensor(dev_forecast)).abs().tolist())
assert t.equal(dev_pairs['copy .7'][0], dev_pairs['zero copy'][0])
assert dev_pairs['zero copy'][1].count_nonzero() == 0


### Worked DEV reflection / self-check
Expected attention is softmax([2,0]) ≈ [.880797,.119203]; copied output is
[.616558,.083442], while uniform selection copies [.35,.35]. A hard-copy forecast
[.7,0] has absolute coordinate errors about [.083442,.083442]. Finite selection,
not failed copying, explains that error. Zero amplitude preserves attention and erases output:
that observation rules out A for this fixture. Self-check: cite *both* the unchanged attention
and changed output, distinguish the uniform-selection control, and say what remains untested.
This does not predict target AUC or learned-model behavior.

### Try It Yourself — changed-condition DEV transfer
Freeze the next change in words: query [1,0] → [0,1], same keys, values, predicate,
sharpness and amplitude. Predict the two output coordinates before the next cell.
Explanation B predicts reversal; an explanation tied to absolute key position does not.


In [ ]:
dev_transfer_forecast = [.083442, .616558]  # Worked exposed forecast; edit first.
dev_transfer_attention, dev_transfer_output = dev_select_copy([0.,1.], .7)
t.testing.assert_close(dev_transfer_output, dev_pairs['copy .7'][1].flip(0))
fig,ax = plt.subplots(figsize=(6,3),layout='constrained')
ax.plot([0,1],dev_pairs['copy .7'][1].tolist(),'o-',label='Original query')
ax.plot([0,1],dev_transfer_output.tolist(),'o-',label='Swapped query')
ax.plot([0,1],dev_pairs['zero copy'][1].tolist(),'--',label='Zero-copy control')
ax.set(xlabel='Output coordinate',ylabel='Activation',title='Observed DEV query transfer',xticks=[0,1]);ax.legend()
display(fig)
print('DEV transfer observed:',dev_transfer_output.tolist())
print('DEV transfer absolute forecast errors:',(dev_transfer_output-t.tensor(dev_transfer_forecast)).abs().tolist())
dev_revision = ''  # Cite the reversal; explain why this tests content selection, not target gain.


## Freeze, then commit
Finish all functions and pass each independent fixture first. Freezing constructs weights,
data and masks for fingerprinting, but makes **no model forward calls**. Read the complete
case printed above, then enter your own numerical forecast, confidence, reasoning and exposure.
The record is immutable JSON text. Rerunning commit creates a separate attempt; preserve the
old variable or export it if desired. Editing methods or runtime afterwards requires a new forecast.
Do not inspect the original answer-rich 8.8 notebook before this step if you want blind practice.


In [ ]:
METHOD_SOURCE = '\n\n'.join([COMPILER_SOURCE, SCORES_SOURCE, AUC_SOURCE, ASSEMBLY_SOURCE])
# The code is student-authored; inspect it before executing.
identity = lesson.freeze_inputs(METHOD_SOURCE)
print('Frozen case, method, helper, environment, tensor and ordered-weight identities.')


In [ ]:
prediction = None  # Your numeric Δ EAP AUC, in [-1,1]
confidence = None  # Unscored reasoning self-rating in [0,1]; not a probability
reasoning = ''  # Explain a causal pathway, and a reason your estimate could fail.
prior_exposure = 'unknown'  # unknown / seen / self_reported_not_seen
commitment = lesson.commit_prediction(identity, METHOD_SOURCE, prediction, confidence, reasoning, prior_exposure)
print('Commitment created. This cell runs no conditions.')


## Observe only after commitment
The next cell explicitly runs three fixed CPU conditions: noisy gain 10, noisy gain 1,
and no-noise gain 1. It scores exact patching, EAP, and IG10 on all valid edges.
One local run has a cooperative 120-second budget checked between stages, one CPU thread,
no downloads/training. A single learner function cannot be forcibly interrupted by this helper;
an over-budget stage is rejected before any successful receipt is returned.
The plotting and replay cells only read stored measurements.
<details><summary>Solution access after commitment</summary>The optional cell below
returns reference method text after validating a commitment. It never modifies your functions.
If you adopt a changed method, freeze again and make a new forecast before running.</details>


In [ ]:
# Optional postcommit reference, no benchmark runs:
# display(Code(lesson.reveal_methods(commitment), language='python'))


In [ ]:
receipt = lesson.run_experiment(commitment, METHOD_SOURCE)
observation = lesson.replay(receipt)
display(lesson.plot_receipt(receipt))
for condition, result in observation['results'].items():
    print(condition, 'accuracy:', result['task_accuracy'], 'mean answer logprob:', result['answer_logprob'])
print('Observed Δ EAP AUC:', observation['delta_eap_auc'])
print('Absolute forecast error:', observation['absolute_forecast_error'])


## Interpret the paired result
Compare the three methods across gains, then the no-noise control. Did changing the output
scale change the task accuracy, its confidence, the ordering of edge scores, or several of
these? Distinguish the derivative at the endpoint from finite single-edge interventions and
the integrated path. Explain both evidence for your causal story and a remaining alternative.
The heatmap shows signed effects on the seven **planted structural edges**. Weight noise can
introduce additional paths: these labels are not complete causal ground truth after noise.
AUC ranks structure; it does not measure circuit completeness or prove scientific correctness.
The horizontal 0.5 line is the expected random-ranking AUC, not a fitted chance result.
The control AUC and task behavior are diagnostics, not an independent replication.


In [ ]:
reflection = ''  # What matched? What changed your causal model? Cite the control.
transfer_question = ''  # Name a changed condition, metric, and fixed denominator.
next_prediction = ''  # Predict it BEFORE testing in separate authorized work.
transfer_confidence = None
transfer_reasoning = ''
# After filling every reflection field, explicitly archive without changing the receipt:
# attempt = lesson.reflect(receipt, reflection, transfer_question, next_prediction, transfer_confidence, transfer_reasoning)
# Try It Yourself: inspect one stored edge without rerunning any condition.
edge = ('input', '1.0<q>')
obs = lesson.replay(receipt)
u, v = obs['sender_order'].index(edge[0]), obs['receiver_order'].index(edge[1])
for condition, result in obs['results'].items():
    print(condition, {name: score[u][v] for name, score in result['scores'].items()})


## Save and replay privately
Export is opt-in and creates a new private file without overwriting. A fresh Python session
can import this helper, read that JSON text, and call `plot_receipt`; no torch, model,
TransformerLens, source methods, environment match, or benchmark execution is required for
receipt replay. Keep your forecast, causal reflection, and experiment receipt together.


In [ ]:
# Optional explicit private export:
# lesson.export_attempt(attempt, Path.home()/'circuit-prediction-attempt-1.json')
# In another session:
# saved = Path.home().joinpath('circuit-prediction-attempt-1.json').read_text()
# archived = lesson.read_attempt(saved)
# print(archived['explanation'], archived['transfer_prediction'])
# display(lesson.plot_receipt(archived['receipt']))


## Limits and connection to real models
This intervention is a small finite-softmax compiled induction transformer. Weights are
written rather than learned; there is no pretrained-model result here. One fixed synthetic
cohort and seed supports a paired diagnostic, not a distributional or real-model claim.
IG10 approximates a path integral; exact single-edge effects depend on the corruption and
background circuit. Redundant paths can have small single-edge effects. Local hashes detect
accidental edits, not a dishonest self-report or modified software. Confidence is recorded;
only absolute numeric forecast error is computed, with no calibration/mastery badge.

RASP and Tracr connect program structure to transformer weights. Read their primary papers
after commitment, alongside the original 8.8 reference. Further investigation could vary
corruption or path discretization under a separate forecast and explicit compute budget:
[RASP](https://arxiv.org/abs/2106.06981), [Tracr](https://arxiv.org/abs/2301.05062).
Human learner walkthrough and scientific acceptance remain deferred.
